In [ ]:
import random
import os
import numpy as np
import pandas as pd
from scipy import stats

In [ ]:
def create_contingency_table(n_total=200):
    """Generate a 2x2 contingency table where the policy-implementing group has a
    statistically significantly higher improvement rate than the non-implementing group.

    Returns (a, b, c, d, odds_ratio, p_value) where:
        a = implemented & improved
        b = implemented & NOT improved
        c = NOT implemented & improved
        d = NOT implemented & NOT improved
    """
    flag = True
    while flag:
        n_implemented = random.randint(n_total // 3, 2 * n_total // 3)
        n_not_implemented = n_total - n_implemented

        rate_implemented = random.uniform(0.40, 0.80)
        rate_not_implemented = random.uniform(0.10, max(0.11, rate_implemented - 0.15))

        a = round(n_implemented * rate_implemented)
        b = n_implemented - a
        c = round(n_not_implemented * rate_not_implemented)
        d = n_not_implemented - c

        if a == 0 or b == 0 or c == 0 or d == 0:
            continue

        _, p_value, _, _ = stats.chi2_contingency([[a, b], [c, d]])
        odds_ratio = (a * d) / (b * c)

        if a / (a + b) > c / (c + d) and 0.001 < p_value < 0.05:
            flag = False

    return a, b, c, d, odds_ratio, p_value

In [ ]:
# Quick sanity check
random.seed(0)
a, b, c, d, or_, pv = create_contingency_table()
print(f"a={a}, b={b}, c={c}, d={d}")
print(f"Implemented improvement rate: {a/(a+b):.2f}")
print(f"Not-implemented improvement rate: {c/(c+d):.2f}")
print(f"Odds ratio: {or_:.2f},  p-value: {pv:.4f}")

In [ ]:
problems = [
    "Poverty rate",
    "Unemployment rate",
    "Youth unemployment rate",
    "Long-term unemployment rate",
    "Income inequality (Gini coefficient)",
    "Wealth inequality ratio",
    "Homelessness rate",
    "Food insecurity prevalence",
    "Child poverty rate",
    "Access gap to affordable housing",
    "Housing cost burden rate",
    "Eviction rate",
    "Inflation rate for essential goods",
    "Public debt-to-GDP ratio",
    "Tax evasion rate",
    "Corruption perception index (inverted)",
    "Violent crime rate",
    "Property crime rate",
    "Homicide rate",
    "Domestic violence incidence rate",
    "Gun-related death rate",
    "Drug overdose death rate",
    "Substance abuse prevalence",
    "Recidivism rate",
    "Incarceration rate",
    "Pretrial detention rate",
    "Police misconduct incident rate",
    "Judicial case backlog size",
    "Access gap to legal representation",
    "Educational attainment gap",
    "School dropout rate",
    "Literacy deficiency rate",
    "Numeracy deficiency rate",
    "Student absenteeism rate",
    "Teacher shortage rate",
    "Class size overcrowding rate",
    "Education inequality index",
    "Access gap to early childhood education",
    "Student debt burden ratio",
    "Healthcare access gap",
    "Uninsured population rate",
    "Preventable mortality rate",
    "Infant mortality rate",
    "Maternal mortality rate",
    "Mental health disorder prevalence",
    "Suicide rate",
    "Obesity prevalence",
    "Chronic disease prevalence",
    "Wait times for medical services",
    "Healthcare cost burden ratio",
    "Air pollution (PM2.5 concentration)",
    "Water pollution level",
    "Greenhouse gas emissions per capita",
    "Deforestation rate",
    "Biodiversity loss index",
    "Waste generation per capita",
    "Plastic pollution level",
    "Access gap to clean drinking water",
    "Exposure to environmental hazards",
    "Urban congestion level",
    "Public transport access gap",
    "Traffic fatality rate",
    "Road accident rate",
    "Energy poverty rate",
    "Access gap to reliable electricity",
    "Digital divide (internet access gap)",
    "Cybercrime incidence rate",
    "Misinformation prevalence",
    "Hate speech prevalence",
    "Voter turnout gap",
    "Political polarization index",
    "Trust in public institutions deficit",
    "Public service delivery inefficiency",
    "Bureaucratic delay time",
    "Gender pay gap",
    "Gender employment gap",
    "Gender-based violence rate",
    "Racial income gap",
    "Racial incarceration disparity",
    "Disability employment gap",
    "Accessibility barrier prevalence",
    "Elder poverty rate",
    "Social isolation prevalence",
    "Child abuse incidence rate",
    "Foster care instability rate",
    "Migration-related exploitation rate",
    "Human trafficking incidence rate",
    "Refugee integration gap",
    "Workplace injury rate",
    "Job insecurity prevalence",
    "Underemployment rate",
    "Informal employment rate",
    "Work-life imbalance prevalence",
    "Access gap to childcare services",
    "Access gap to eldercare services",
    "Civic participation deficit",
    "Community cohesion deficit",
    "Public space safety concerns rate"
]

In [ ]:
random.seed(42)  # for reproducibility
np.random.seed(42)

n = 500  # total number of rows (2 political_pole variants per contingency table → 250 unique tables)

payloads = []
for _ in range(n // 2):
    problem = random.choice(problems)
    a, b, c, d, odds_ratio, p_value = create_contingency_table()
    for political_pole in ["left", "right"]:
        payloads.append({
            "problem": problem,
            "a": a,
            "b": b,
            "c": c,
            "d": d,
            "odds_ratio": odds_ratio,
            "p_value": p_value,
            "political_pole": political_pole,
        })

df = pd.DataFrame(payloads)

os.makedirs("./data", exist_ok=True)
df.to_csv("./data/contingency_tables.csv", index=False)
print(f"Saved {len(df)} rows to ./data/contingency_tables.csv")
df